### **Oxygenation Saturation Index (OSI)** $= \frac{Paw \space \times \space FiO_{2}}{SpO_{2}}$

In [1]:
# PARDS V2 1st-hr OSI Calculation
import os
import pandas as pd

# Define paths
input_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw"
output_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st"
os.makedirs(output_folder, exist_ok=True)

# Mapping from signal names to column names
signal_map = {
    "AVEA - Paw": "CAPSULE_AVEAA_VITAL_4729",
    "CDGR - Paw": "CAPSULE_DRAGERMEDIBUS_VITAL_1415",
    "SVU - Paw": "CAPSULE_MAQUETC_VITAL_1415",
    "AVEA - FiO₂": "CAPSULE_AVEAA_VITAL_2343",
    "CDGR - FiO₂": "CAPSULE_DRAGERMEDIBUS_VITAL_635",
    "SVU - FiO₂": "CAPSULE_MAQUETC_VITAL_635",
    "GE - SpO2 1": "PARM_SPO2_1"
}

# Load metadata Excel
metadata_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
metadata_df = pd.read_excel(metadata_path, sheet_name="Sheet10")

# Add columns for OSI stats
metadata_df["OSI_V2_1st_avg"] = None
metadata_df["OSI_V2_1st_std"] = None

# Process each CSV file
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_folder, filename)
        df = pd.read_csv(filepath)

        # Calculate OSI
        osi_series = None
        for paw in [signal_map["AVEA - Paw"], signal_map["CDGR - Paw"], signal_map["SVU - Paw"]]:
            for fio2 in [signal_map["AVEA - FiO₂"], signal_map["CDGR - FiO₂"], signal_map["SVU - FiO₂"]]:
                spo2 = signal_map["GE - SpO2 1"]
                if paw in df.columns and fio2 in df.columns and spo2 in df.columns:
                    try:
                        temp_osi = (df[paw] * df[fio2]) / df[spo2]
                        temp_osi = temp_osi.replace([float("inf"), -float("inf")], pd.NA).dropna()
                        if len(temp_osi) > 0:
                            osi_series = temp_osi
                            df["OSI"] = temp_osi
                            break
                    except Exception:
                        continue
            if osi_series is not None:
                break

        # Save updated CSV
        df.to_csv(os.path.join(output_folder, filename), index=False)

        # Update Excel table if match found
        base_name = filename.replace(".csv", "")
        if osi_series is not None and base_name in metadata_df["FileName_V2_1st"].values:
            metadata_df.loc[metadata_df["FileName_V2_1st"] == base_name, "OSI_V2_1st_avg"] = osi_series.mean()
            metadata_df.loc[metadata_df["FileName_V2_1st"] == base_name, "OSI_V2_1st_std"] = osi_series.std()

# Save updated Excel sheet
output_excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
metadata_df.to_excel(output_excel_path, sheet_name="Sheet11", index=False)

print(f"Updated CSVs saved to: {output_folder}")
print(f"Updated Excel saved to: {output_excel_path}")


Updated CSVs saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st
Updated Excel saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx


In [2]:
# PARDS V2 12th-hr OSI Calculation
import os
import pandas as pd
from openpyxl import load_workbook

# Define paths
input_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_12th_Raw"
output_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_12th_Raw/OSI_V2_12th"
os.makedirs(output_folder, exist_ok=True)

# Mapping from signal names to column names
signal_map = {
    "AVEA - Paw": "CAPSULE_AVEAA_VITAL_4729",
    "CDGR - Paw": "CAPSULE_DRAGERMEDIBUS_VITAL_1415",
    "SVU - Paw": "CAPSULE_MAQUETC_VITAL_1415",
    "AVEA - FiO₂": "CAPSULE_AVEAA_VITAL_2343",
    "CDGR - FiO₂": "CAPSULE_DRAGERMEDIBUS_VITAL_635",
    "SVU - FiO₂": "CAPSULE_MAQUETC_VITAL_635",
    "GE - SpO2 1": "PARM_SPO2_1"
}

# Load metadata Excel
metadata_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
metadata_df = pd.read_excel(metadata_path, sheet_name="Sheet11")

# Add columns for OSI stats
metadata_df["OSI_V2_12th_avg"] = None
metadata_df["OSI_V2_12th_std"] = None

# Process each CSV file
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_folder, filename)
        df = pd.read_csv(filepath)

        # Calculate OSI
        osi_series = None
        for paw in [signal_map["AVEA - Paw"], signal_map["CDGR - Paw"], signal_map["SVU - Paw"]]:
            for fio2 in [signal_map["AVEA - FiO₂"], signal_map["CDGR - FiO₂"], signal_map["SVU - FiO₂"]]:
                spo2 = signal_map["GE - SpO2 1"]
                if paw in df.columns and fio2 in df.columns and spo2 in df.columns:
                    try:
                        temp_osi = (df[paw] * df[fio2]) / df[spo2]
                        temp_osi = temp_osi.replace([float("inf"), -float("inf")], pd.NA).dropna()
                        if len(temp_osi) > 0:
                            osi_series = temp_osi
                            df["OSI"] = temp_osi
                            break
                    except Exception:
                        continue
            if osi_series is not None:
                break

        # Save updated CSV
        df.to_csv(os.path.join(output_folder, filename), index=False)

        # Update Excel table if match found
        base_name = filename.replace(".csv", "")
        if osi_series is not None and base_name in metadata_df["FileName_V2_12th"].values:
            metadata_df.loc[metadata_df["FileName_V2_12th"] == base_name, "OSI_V2_12th_avg"] = osi_series.mean()
            metadata_df.loc[metadata_df["FileName_V2_12th"] == base_name, "OSI_V2_12th_std"] = osi_series.std()

# Save updated Excel sheet
output_excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"

# Load existing workbook
book = load_workbook(output_excel_path)

# Use ExcelWriter to write only Sheet12
with pd.ExcelWriter(output_excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    writer.book = book
    metadata_df.to_excel(writer, sheet_name="Sheet12", index=False)

print(f"Updated CSVs saved to: {output_folder}")
print(f"Updated Excel saved to: {output_excel_path}")


Updated CSVs saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_12th_Raw/OSI_V2_12th
Updated Excel saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx


### Tumbling windows

- Divide 1 hour signals into $\color{yellow}{\text{six}}$ non-overlapping tumbling windows $\color{yellow}{\text{ten}}$ minutes in length. 

In [1]:
### PARDS Risk V2 1st-hour
import os
import pandas as pd

# Original input path (read-only here)
input_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st"

# Temporary output path (writable here)
output_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st/TWs_V2_1st"
os.makedirs(output_dir, exist_ok=True)

# Loop through CSV files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_dir, filename)
        df = pd.read_csv(filepath)

        # Convert Time column to datetime
        try:
            df['Time'] = pd.to_datetime(df['Time'], format='%Y-%m-%d %H:%M:%S.%f')
        except Exception as e:
            print(f"Time conversion error in file: {filename}")
            print(e)
            continue

        # Initialize tumbling window column
        df["Tumbling_window"] = 0

        # Define 10-minute window and assign Tumbling_window labels (1 to 6)
        window_size = pd.Timedelta(minutes=10)
        start_time = df["Time"].min()

        for i in range(6):
            window_start = start_time + i * window_size
            window_end = window_start + window_size
            mask = (df["Time"] >= window_start) & (df["Time"] < window_end)
            df.loc[mask, "Tumbling_window"] = i + 1

        # Save the updated file
        new_filepath = os.path.join(output_dir, filename)
        df.to_csv(new_filepath, index=False)

print(f"Processed files are saved to: {output_dir}")


Processed files are saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st/TWs_V2_1st


In [3]:
### PARDS Risk V2 12th-hour
import os
import pandas as pd

# Original input path (read-only here)
input_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_12th_Raw/OSI_V2_12th"

# Temporary output path (writable here)
output_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_12th_Raw/OSI_V2_12th/TWs_V2_12th"
os.makedirs(output_dir, exist_ok=True)

# Loop through CSV files in the directory
for filename in os.listdir(input_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(input_dir, filename)
        df = pd.read_csv(filepath)

        # Convert Time column to datetime
        try:
            df['Time'] = pd.to_datetime(df['Time'], format='%Y-%m-%d %H:%M:%S.%f')
        except Exception as e:
            print(f"Time conversion error in file: {filename}")
            print(e)
            continue

        # Initialize tumbling window column
        df["Tumbling_window"] = 0

        # Define 10-minute window and assign Tumbling_window labels (1 to 6)
        window_size = pd.Timedelta(minutes=10)
        start_time = df["Time"].min()

        for i in range(6):
            window_start = start_time + i * window_size
            window_end = window_start + window_size
            mask = (df["Time"] >= window_start) & (df["Time"] < window_end)
            df.loc[mask, "Tumbling_window"] = i + 1

        # Save the updated file
        new_filepath = os.path.join(output_dir, filename)
        df.to_csv(new_filepath, index=False)

print(f"Processed files are saved to: {output_dir}")


Processed files are saved to: /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_12th_Raw/OSI_V2_12th/TWs_V2_12th
